In [ ]:
# Copyright (c) 2025 Microsoft Corporation.
import os
from pathlib import Path
from typing import cast

import numpy as np
import pandas as pd
from pydantic import SecretStr
from rich import print as rich_print

from benchmark_qed.autoe.assertion_scores import get_assertion_scores
from benchmark_qed.autoe.visualization.assertions import (
    plot_assertion_accuracy_by_rag_method,
)
from benchmark_qed.cli.utils import print_df
from benchmark_qed.config.llm_config import LLMConfig, LLMProvider
from benchmark_qed.llm.factory import ModelFactory

# AutoE: Assertion-based Scoring

Assertion-based scoring evaluates RAG-generated answers by checking whether they satisfy specific factual statements (assertions) that should be present in a correct response. This approach is especially useful when the presence or absence of key facts matters more than holistic answer quality.

This notebook demonstrates how to:
1. Evaluate multiple RAG methods against a set of assertions across multiple question sets
2. Aggregate and compare per-method assertion pass rates
3. Visualize the results to compare RAG methods at a glance

Assertions can be generated automatically using AutoQ (see the [AutoQ notebook](autoq.ipynb) or the [Assertion Generation notebook](assertion_gen.ipynb)) or defined manually.

In [ ]:
import nest_asyncio

nest_asyncio.apply()

In [ ]:
%load_ext dotenv
%dotenv

## Configuration

In [ ]:
# Config LLM model to be used as judge
llm_config = LLMConfig(
    model="gpt-4.1",
    api_key=SecretStr(os.environ["OPENAI_API_KEY"]),
    llm_provider=LLMProvider.OpenAIChat,
    concurrent_requests=100,
    call_args={"temperature": 0.0, "seed": 42},
)
llm_client = ModelFactory.create_chat_model(llm_config)

In [ ]:
# Config conditions for evaluation
generated_rags = ["vector_rag", "lazygraphrag", "graphrag_global"]
question_sets = ["activity_global", "activity_local"]
trials = 4  # number of trials for each (question, assertion) pair. Trials must be an even number to support counterbalancing.
pass_threshold = 0.5  # fraction of trials that must pass for an assertion to be considered passing

input_dir = "./example_answers"
output_dir = Path("./output/assertion_scores")
output_dir.mkdir(parents=True, exist_ok=True)

## Assertion-based Evaluation

For each combination of RAG method and question set, we:
1. Load the RAG-generated answers and the corresponding assertions
2. Use the LLM judge to determine whether each answer satisfies each assertion
3. Aggregate results into per-question and per-method summaries

In [ ]:
all_results = []

for question_set in question_sets:
    rich_print(f"Processing question set: [bold]{question_set}[/bold]")

    # Load assertions for this question set
    assertions = (
        pd.read_json(f"{input_dir}/{question_set}_assertions.json")
        .explode("assertions")
        .rename(columns={"assertions": "assertion"})
        .reset_index(drop=True)
    )

    rich_print(f"  Loaded {len(assertions)} assertions")

    for generated_rag in generated_rags:
        rich_print(f"  Evaluating [bold]{generated_rag}[/bold]...")

        answers_path = f"{input_dir}/{generated_rag}/{question_set}.json"
        answers = pd.read_json(answers_path)

        # Score assertions for this RAG method
        assertion_score = get_assertion_scores(
            llm_client=llm_client,
            llm_config=llm_config,
            answers=answers,
            assertions=assertions,
            trials=trials,
            question_id_key="question_id",
            question_text_key="question_text",
            answer_text_key="answer",
        )

        # Save raw scores for this method and question set
        qs_output_dir = output_dir / question_set
        qs_output_dir.mkdir(parents=True, exist_ok=True)
        assertion_score.to_csv(
            qs_output_dir / f"{generated_rag}_assertion_scores.csv", index=False
        )

        # Compute pass/fail per (question, assertion) pair
        summary_by_assertion = (
            assertion_score.groupby(["question", "assertion"])
            .agg(
                score=("score", lambda x: int(x.mean() > pass_threshold)),
                score_mean=("score", "mean"),
                score_std=("score", "std"),
            )
            .reset_index()
        )

        # Compute per-question pass/fail counts
        summary_by_question = (
            summary_by_assertion.groupby(["question"])
            .agg(
                success=("score", lambda x: (x == 1).sum()),
                fail=("score", lambda x: (x == 0).sum()),
            )
            .reset_index()
        )

        # Save per-question and per-assertion summaries
        summary_by_question.to_csv(
            qs_output_dir / f"{generated_rag}_summary_by_question.csv", index=False
        )
        summary_by_assertion.to_csv(
            qs_output_dir / f"{generated_rag}_summary_by_assertion.csv", index=False
        )

        # Compute overall accuracy for this method/question set
        total_success = summary_by_question["success"].sum()
        total_fail = summary_by_question["fail"].sum()
        total_assertions = total_success + total_fail
        overall_accuracy = (
            total_success / total_assertions if total_assertions > 0 else 0.0
        )

        # Report failed assertions
        failed_assertions = cast(
            pd.DataFrame, summary_by_assertion[summary_by_assertion["score"] == 0]
        )
        if len(failed_assertions) > 0:
            rich_print(
                f"    [bold red]{generated_rag}: {len(failed_assertions)} assertions failed[/bold red]"
            )
        else:
            rich_print(
                f"    [bold green]{generated_rag}: All assertions passed[/bold green]"
            )
        rich_print(
            f"    Overall accuracy: {overall_accuracy:.3f} ({total_success}/{total_assertions})"
        )

        all_results.append(
            {
                "question_set": question_set,
                "rag_method": generated_rag,
                "total_assertions": total_assertions,
                "successful_assertions": total_success,
                "failed_assertions": total_fail,
                "overall_accuracy": overall_accuracy,
                "total_questions": len(summary_by_question),
            }
        )

rich_print("[bold green]Assertion evaluation complete.[/bold green]")

## Results Summary

The table below shows the overall assertion-based accuracy for each RAG method and question set.

In [ ]:
overall_summary_df = pd.DataFrame(all_results)
overall_summary_df = overall_summary_df.sort_values(
    ["question_set", "overall_accuracy"], ascending=[True, False]
).reset_index(drop=True)

# Save overall summary
overall_summary_df.to_csv(output_dir / "assertion_scores_overall_summary.csv", index=False)

print_df(
    overall_summary_df,
    "Overall Assertion Scores Summary by Question Set and RAG Method",
)

# Also show pivot table for easy cross-method comparison
pivot_summary = overall_summary_df.pivot_table(
    index="rag_method", columns="question_set", values="overall_accuracy"
).reset_index()
pivot_summary.to_csv(output_dir / "assertion_scores_pivot_summary.csv", index=False)

print_df(
    pivot_summary,
    "Assertion Accuracy Comparison (Pivot View)",
)

## Visualization

The chart below compares assertion-based accuracy across all evaluated RAG methods and question sets.

In [ ]:
fig, ax = plot_assertion_accuracy_by_rag_method(
    results_df=overall_summary_df,
    output_path=output_dir / "assertion_accuracy_chart.png",
    title="Assertion-based Accuracy by RAG Method and Question Set",
)
rich_print("Model usage statistics:")
rich_print(llm_client.get_usage())